# 01 — Exploratory Data Analysis

Dataset: `ajgt_twitter_ar` — 1800 Arabic tweets, binary sentiment (0=negative, 1=positive)

Questions to answer here:
- What does the raw data actually look like?
- Any nulls, duplicates, or garbage rows?
- How long are the tweets? (affects model truncation later)
- Is the class distribution balanced?
- What are some real examples from each class?

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/raw/ajgt_twitter_ar.csv')
df.head(10)

,text,label
0,اربد فيها جامعات اكثر من عمان ... وفيها قد عم...,1
1,الحلو انكم بتحكوا على اساس انو الاردن ما فيه ...,0
2,كله رائع بجد ربنا يكرمك,1
3,لسانك قذر يا قمامه,0
4,​انا داشره وغير متزوجه ولدي علاقات مشبوه واحشش...,0
5,ابشرك فيه تحسن ولله الحمد باذن الله يرجع قريبا,1
6,ابو الشباب راعي العود ليش ماوزنه في البيت غباء,0
7,ابو معيتق قطع اوتار العود وقال السلام عليكم,0
8,اتحزن فان الله يدافع عنك والملائكه تستغفر لك و...,1
9,اترك ما تهوى لاجل من تخشى,1


In [2]:
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.info()

Shape: (1800, 2)
Columns: ['text', 'label']

<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    1800 non-null   str  
 1   label   1800 non-null   int64
dtypes: int64(1), str(1)
memory usage: 178.5 KB


In [3]:
# Null check
df.isnull().sum()

text     0
label    0
dtype: int64

In [4]:
# Duplicate check
print(f'Duplicates: {df.duplicated().sum()}')

Duplicates: 0


In [5]:
# Tweet length in characters
df['char_count'] = df['text'].str.len()

fig = px.histogram(
    df, x='char_count', nbins=50,
    color='label',
    title='Tweet length distribution by class',
    labels={'char_count': 'characters', 'label': 'sentiment (0=neg, 1=pos)'},
    barmode='overlay',
    opacity=0.7
)
fig.show()

print(df['char_count'].describe().round(1))

count    1800.0
mean       46.8
std        46.1
min         6.0
25%        23.0
50%        36.0
75%        58.0
max       864.0
Name: char_count, dtype: float64


In [6]:
# Labels are integers — map to readable strings before anything else
df['sentiment'] = df['label'].map({1: 'positive', 0: 'negative'})

counts = df['sentiment'].value_counts().reset_index()
fig = px.bar(
    counts, x='sentiment', y='count',
    color='sentiment',
    color_discrete_map={'positive': '#2ecc71', 'negative': '#e74c3c'},
    title='Class distribution',
)
fig.update_layout(showlegend=False)
fig.show()

print(df['sentiment'].value_counts())

sentiment
positive    900
negative    900
Name: count, dtype: int64


## Sample tweets — read these manually

This is important. Before running any model, read actual examples from each class.
Ask yourself: does the label make sense? Are there obvious mistakes? What patterns do you notice?

In [7]:
pd.set_option('display.max_colwidth', 200)

print('=== POSITIVE SAMPLES ===')
display(df[df['sentiment'] == 'positive'][['text']].sample(5, random_state=42))

print('=== NEGATIVE SAMPLES ===')
display(df[df['sentiment'] == 'negative'][['text']].sample(5, random_state=42))

=== POSITIVE SAMPLES ===


,text
153,الجمال : ليس فقط شيئا نراه بل هو شيء نكتشفه ، روح جميله ، وفكر جميل ، اخلاق جميله ، وادب جميل
1683,يا ارحم الراحمين يا الله
376,اللهم انا نسالك رزقا حلال طيبا مباركا يعيننا علي طاعتك
1019,على الاقل هناك رجل وفي لامراه احبها وهذا يسعدني
79,اصدقاء اوفياء ونتعلم منهم


=== NEGATIVE SAMPLES ===


,text
128,الاردن كان افضل حالا بمئات المرات في التعليم و الصحه و الزراعه و الاستثمار ... التردي ليس مرتبط بالربيع العربي بل قبله بمده عام
1643,وش يحسووون فيه ناس مهبل
588,باللـــه عليكم الم تستحيوا من انفسكم قبل قدومكم الى هذا البرنامج ثم الم تعلموا ان هناك مسابقات لانكر الاصوات في قناه اخرى
1292,ليس هناك شخص معاق بل هناك مجتمع يعيق
77,اشي بقهر مش لا هل درجه


In [8]:
# Word count per tweet
df['word_count'] = df['text'].str.split().str.len()

print('Word count stats:')
print(df.groupby('sentiment')['word_count'].describe().round(1))

# Shortest tweets — likely garbage or edge cases
print('\nShortest 5 tweets:')
display(df.nsmallest(5, 'word_count')[['text', 'sentiment', 'word_count']])

Word count stats:
           count  mean   std  min  25%  50%   75%    max
sentiment                                               
negative   900.0   9.2  10.3  1.0  4.0  7.0  11.0  143.0
positive   900.0   8.8   5.5  2.0  5.0  7.0  11.0   33.0

Shortest 5 tweets:


,text,sentiment,word_count
1354,مايستحون,negative,1
33,اخبار مسخره,negative,2
64,استهبال باستهبال,negative,2
105,اقرفني والله,negative,2
117,اكيد بقرف,negative,2


## Findings

Write what you notice after running the cells above. Some prompts:

- Does tweet length differ between positive and negative? Why might that matter?
- Did any of the sample tweets surprised you — label seems wrong, or text is ambiguous?
- Any rows you'd want to drop before cleaning? (very short, garbled, mostly URLs)
- What's the distribution telling you about how the dataset was collected?

*(fill in after running)*